# The Chat Format

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [ ]:
!pip install python-dotenv
!pip install openai==0.28.0

!pip install jupyter_bokeh

!pip install -q --upgrade panel
!pip install jupyter_bokeh

In [ ]:
# one-cell setup: pulls the key you stored in “Notebook access”
from google.colab import userdata
import openai, sys

api_key = userdata.get("OPENAI_API_KEY1")          # ← name must match the secret
if not api_key:
    sys.exit("❌  OPENAI_API_KEY not available. Click the padlock ➜ toggle it on, then rerun.")

openai.api_key = api_key
print("✅ OpenAI API key loaded.")


✅ OpenAI API key loaded.


In [ ]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]

def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
#     print(str(response.choices[0].message))
    return response.choices[0].message["content"]

In [ ]:
messages =  [
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
{'role':'user', 'content':'tell me a joke'},
{'role':'assistant', 'content':'Why did the chicken cross the road'},
{'role':'user', 'content':'I don\'t know'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Verily, the chicken crossed the road to reach the other side, forsooth!


In [ ]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is umar'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's great to meet you. How are you doing today?


In [ ]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, I don't have the ability to remember user names or personal information. How can I assist you today?


In [ ]:

messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is umar'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Umar.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [ ]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [ ]:
import os, openai, panel as pn
from dotenv import load_dotenv, find_dotenv

SYSTEM_PROMPT = """You are OrderBot … (menu here) …"""
conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

pn.extension()

chat_log   = pn.pane.Markdown("", height=300)               # ← no scroll=…
user_input = pn.widgets.TextInput(placeholder="Type here…")
send_btn   = pn.widgets.Button(name="Send", button_type="primary")

def _render():
    md = []
    for msg in conversation[1:]:
        speaker = "🧑" if msg["role"]=="user" else "🤖"
        md.append(f"**{speaker} {msg['role'].capitalize()}**: {msg['content']}")
    chat_log.object = "\n\n".join(md)

def on_send(_):
    text = user_input.value.strip()
    if not text: return
    conversation.append({"role": "user", "content": text})
    _render()
    user_input.value = ""

    reply = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=conversation,
        temperature=0.7
    )["choices"][0]["message"]["content"]

    conversation.append({"role": "assistant", "content": reply})
    _render()

send_btn.on_click(on_send)

app = pn.Column(chat_log, pn.Row(user_input, send_btn), width=600)
app.servable()           # use `panel serve` or just display `app` in-notebook


Column(width=600)
    [0] Markdown(str, height=300)
    [1] Row
        [0] TextInput(placeholder='Type here…')
        [1] Button(button_type='primary', name='Send')

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response = get_completion_from_messages(messages, temperature=0)
print(response)

NameError: name 'context' is not defined

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!